# TP 5 — NLP : TF-IDF, baseline de classification et décodage

> Support théorique : [cours_nlp_et_llms.md](cours_nlp_et_llms.md).

**Objectif.** Mettre en œuvre, hors ligne et sans téléchargement, les idées clés du module 5 :
représenter du texte avec **TF-IDF**, entraîner une **baseline solide** (TF-IDF + régression logistique),
mesurer une **similarité cosinus** entre documents, lire les **mots discriminants** d'un modèle linéaire,
et manipuler la **température de décodage**.

**Note pédagogique.** Le corpus est minuscule et synthétique : il sert à rendre les mécanismes visibles,
pas à établir une performance. Le protocole du module 2 (split, baseline, métrique) reste la règle.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(0)
print("Environnement prêt.")

Environnement prêt.


## 1. Un petit corpus étiqueté

Deux classes d'avis clients : `1` = positif, `0` = négatif. Le texte est une donnée **non structurée**
(module 1) : on le transforme en nombres avant tout modèle.

In [2]:
avis = [
    ("livraison rapide et produit parfait je recommande", 1),
    ("excellent service tout est parfait merci", 1),
    ("article génial conforme et de bonne qualité", 1),
    ("très satisfait rapide et efficace bravo", 1),
    ("parfait rien à redire je recommande vivement", 1),
    ("colis rapide article parfait et bien emballé", 1),
    ("qualité au rendez vous vendeur sérieux et rapide", 1),
    ("super produit conforme livraison rapide", 1),
    ("génial je suis satisfait et je recommande", 1),
    ("produit excellent emballage parfait service au top", 1),
    ("expérience parfaite rapide et sans souci", 1),
    ("très bonne qualité conforme et rapide bravo", 1),
    ("cassé à l arrivée service injoignable horrible", 0),
    ("produit décevant lent et de mauvaise qualité", 0),
    ("arnaque total remboursement refusé service horrible", 0),
    ("livraison lente et article défectueux déçu", 0),
    ("mauvaise qualité cassé je déconseille fortement", 0),
    ("horrible expérience produit non conforme et lent", 0),
    ("déçu article abîmé et service client injoignable", 0),
    ("remboursement refusé arnaque à éviter absolument", 0),
    ("produit défectueux lent horrible je déconseille", 0),
    ("mauvaise surprise cassé et de piètre qualité", 0),
    ("service lamentable lent et article non conforme", 0),
    ("déçu par la qualité produit cassé et injoignable", 0),
]
textes = [t for t, _ in avis]
y = np.array([label for _, label in avis])
print(f"{len(textes)} avis, {y.sum()} positifs / {(y==0).sum()} négatifs")

X_tr, X_te, y_tr, y_te = train_test_split(
    textes, y, test_size=0.33, random_state=0, stratify=y
)
print(f"train : {len(X_tr)} | test : {len(X_te)}")

24 avis, 12 positifs / 12 négatifs
train : 16 | test : 8


## 2. Baseline TF-IDF + régression logistique

Le TF-IDF abaisse le poids des mots omniprésents et relève celui des mots discriminants (module 5, §3.2).
On l'enchaîne à une régression logistique dans une **Pipeline** : le vocabulaire et les poids IDF sont
appris **uniquement sur le train** (pas de fuite, module 1).

In [3]:
modele = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 1), min_df=1),
    LogisticRegression(max_iter=1000, random_state=0),
)
modele.fit(X_tr, y_tr)

y_pred = modele.predict(X_te)
acc = accuracy_score(y_te, y_pred)
f1 = f1_score(y_te, y_pred)

# Baseline triviale : toujours prédire la classe majoritaire du train (module 2, §7.1)
classe_majoritaire = int(round(y_tr.mean()))
acc_baseline = accuracy_score(y_te, np.full_like(y_te, classe_majoritaire))

print(f"Accuracy modèle   : {acc:.3f}")
print(f"Accuracy baseline : {acc_baseline:.3f} (classe majoritaire)")
print(f"F1 (classe positive) : {f1:.3f}")
print()
print(classification_report(y_te, y_pred, target_names=["négatif", "positif"]))
assert acc > acc_baseline, "le modèle doit battre la baseline de classe majoritaire"
assert acc >= 0.7, "sur ce corpus séparable, l'accuracy de test doit rester élevée" 

Accuracy modèle   : 1.000
Accuracy baseline : 0.500 (classe majoritaire)
F1 (classe positive) : 1.000

              precision    recall  f1-score   support

     négatif       1.00      1.00      1.00         4
     positif       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



## 3. Similarité cosinus entre documents

Deux avis qui partagent des mots ont des vecteurs TF-IDF proches (cosinus élevé), même formulés
différemment. On compare une paire **proche** et une paire **éloignée** (module 0, §2.3 ; module 5, §3.3).

In [4]:
phrases = [
    "livraison rapide et produit parfait",   # 0
    "colis rapide article parfait",          # 1  (proche de 0)
    "service horrible et remboursement refusé",  # 2 (éloignée)
]
vect = TfidfVectorizer().fit(textes)         # vocabulaire du corpus complet (démonstration)
V = vect.transform(phrases)
S = cosine_similarity(V)
print("Matrice de similarité cosinus :")
print(np.round(S, 3))

sim_proche = S[0, 1]
sim_eloignee = S[0, 2]
print(f"\ncos(0, 1) proche   = {sim_proche:.3f}")
print(f"cos(0, 2) éloignée = {sim_eloignee:.3f}")
assert sim_proche > sim_eloignee, "la paire partageant du vocabulaire doit être plus proche" 

Matrice de similarité cosinus :
[[1.    0.389 0.072]
 [0.389 1.    0.   ]
 [0.072 0.    1.   ]]

cos(0, 1) proche   = 0.389
cos(0, 2) éloignée = 0.072


## 4. Quels mots le modèle a-t-il retenus ?

Un modèle linéaire sur TF-IDF est **interprétable** : le signe et l'amplitude du coefficient d'un mot
indiquent vers quelle classe il pousse (module 2, §13 ; module 8, §3).

In [5]:
vectorizer = modele.named_steps["tfidfvectorizer"]
clf = modele.named_steps["logisticregression"]
mots = np.array(vectorizer.get_feature_names_out())
poids = clf.coef_[0]                     # >0 pousse vers 'positif', <0 vers 'négatif'

ordre = np.argsort(poids)
print("Top mots -> POSITIF :", list(mots[ordre[-6:]][::-1]))
print("Top mots -> NÉGATIF :", list(mots[ordre[:6]]))
# Contrôle de cohérence : un mot clairement positif doit avoir un poids positif.
assert poids[list(mots).index("parfait")] > 0
assert poids[list(mots).index("horrible")] < 0

Top mots -> POSITIF : ['rapide', 'parfait', 'livraison', 'bonne', 'recommande', 'très']
Top mots -> NÉGATIF : ['lent', 'mauvaise', 'déconseille', 'horrible', 'cassé', 'injoignable']


## 5. Décodage : l'effet de la température

À la génération, la **température** aplatit ou pique la distribution avant l'échantillonnage
(module 5, §8). Une température basse → sortie plus déterministe (entropie faible) ; une température
haute → plus de diversité (entropie élevée).

In [6]:
def softmax(logits, temperature=1.0):
    z = np.asarray(logits, dtype=float) / temperature
    z -= z.max()                          # stabilité numérique (module 0, §8.4)
    e = np.exp(z)
    return e / e.sum()

def entropie(p):
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

logits = np.array([2.0, 1.0, 0.2, -0.5, -1.0])   # un token nettement favori
for T in (0.5, 1.0, 2.0):
    p = softmax(logits, T)
    print(f"T={T}: distribution={np.round(p, 3)}  entropie={entropie(p):.3f}")

e_basse = entropie(softmax(logits, 0.5))
e_moyenne = entropie(softmax(logits, 1.0))
e_haute = entropie(softmax(logits, 2.0))
assert e_basse < e_moyenne < e_haute, "l'entropie croît avec la température"
print("\n-> plus la température monte, plus la distribution est plate (choix plus diversifié).")

T=0.5: distribution=[0.853 0.115 0.023 0.006 0.002]  entropie=0.515
T=1.0: distribution=[0.601 0.221 0.099 0.049 0.03 ]  entropie=1.122
T=2.0: distribution=[0.396 0.24  0.161 0.114 0.088]  entropie=1.465

-> plus la température monte, plus la distribution est plate (choix plus diversifié).


## 6. Exercice guidé — classer un nouvel avis

Écrivez `classer(texte)` qui renvoie `(label, probabilite_positif)` à partir du `modele` entraîné.
**Critère de réussite** : les deux exemples de test ci-dessous doivent recevoir le bon label.

In [7]:
def classer(texte):
    # TODO : utiliser modele.predict([...]) et modele.predict_proba([...])[:, 1]
    # Renvoyer (int(label), float(probabilite_de_la_classe_positive))
    raise NotImplementedError("À compléter")

# Décommentez pour tester :
# assert classer("produit parfait et livraison rapide")[0] == 1
# assert classer("horrible service et article cassé")[0] == 0

### Solution

In [8]:
def classer(texte):
    label = int(modele.predict([texte])[0])
    proba_pos = float(modele.predict_proba([texte])[0, 1])
    return label, proba_pos

ex_pos = classer("produit parfait et livraison rapide")
ex_neg = classer("horrible service et article cassé")
print("positif attendu ->", ex_pos)
print("négatif attendu ->", ex_neg)
assert ex_pos[0] == 1 and ex_neg[0] == 0
print("OK : les deux avis sont correctement classés.")

positif attendu -> (1, 0.6623148823359818)
négatif attendu -> (0, 0.39440167155772526)
OK : les deux avis sont correctement classés.


## Récapitulatif

- **TF-IDF + régression logistique** est une baseline solide à battre avant tout modèle lourd.
- La **similarité cosinus** rapproche les documents partageant du vocabulaire (base du RAG, module 4).
- Un modèle linéaire est **interprétable** : on lit les mots qui poussent vers chaque classe.
- La **température** contrôle la diversité du décodage.

Attention : ce corpus est un jouet. Sur de vraies données, gérez déséquilibre, quasi-doublons,
contamination et **hallucination** pour les parties génératives (module 5, §9).

Suite : [Module 6 — apprentissage par renforcement](../06_apprentissage_par_renforcement/cours_apprentissage_par_renforcement.md).